## Installation


In [ ]:
%pip install "gymnasium[classic-control]"

## 2. Imports
- `gymnasium` — the RL environment toolkit itself
- `numpy` — for arrays and numerical operations (Gymnasium observations come back as NumPy arrays)
- `matplotlib.pyplot` — for plotting our agent's learning progress, and for building our video clips


In [8]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

SEED = 0
np.random.seed(SEED)

## Creating the Environment
We create environments with `gym.make(environment_id)`. The id is just a string Gymnasium looks up in its registry — `"Acrobot-v1"` here.


In [9]:
env = gym.make("Acrobot-v1")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<AcrobotEnv<Acrobot-v1>>>>>

`env` is now an object representing the Acrobot world. We haven't started an episode yet — for that, we need `reset()`.

### Python refresher: tuple unpacking
`env.reset()` returns **two things at once**: the starting observation, and an `info` dictionary (usually empty, used for debugging/extra info). In Python, when a function returns multiple values, you can "unpack" them directly into separate variables in one line:

```python
observation, info = env.reset()
```

This is the exact same trick as:
```python
x, y = 3, 4
```
Python just matches up the items on the right with the names on the left, in order.


In [10]:
observation, info = env.reset(seed=SEED)

print("Observation:", observation)
print("Info:", info)

Observation: [ 0.99962485  0.02738891  0.9989402  -0.04602639 -0.09180529 -0.09669447]
Info: {}


In [11]:
print("Observation space:", env.observation_space)
print("Shape:", env.observation_space.shape)
print("Lower bounds:", env.observation_space.low)
print("Upper bounds:", env.observation_space.high)


Observation space: Box([ -1.        -1.        -1.        -1.       -12.566371 -28.274334], [ 1.        1.        1.        1.       12.566371 28.274334], (6,), float32)
Shape: (6,)
Lower bounds: [ -1.        -1.        -1.        -1.       -12.566371 -28.274334]
Upper bounds: [ 1.        1.        1.        1.       12.566371 28.274334]


In [12]:
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)


# .sample() picks a uniformly random valid action
print("A random action:", env.action_space.sample())

Action space: Discrete(3)
Number of actions: 3
A random action: 1


In [13]:
observation, info = env.reset(seed=SEED)

action = env.action_space.sample()  # pick a random action
observation, reward, terminated, truncated, info = env.step(action)

print("Action taken:", action)
print("New observation:", observation)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)


Action taken: 1
New observation: [ 0.9999849   0.00548956  0.9983176  -0.05798252 -0.12309824 -0.02443151]
Reward: -1.0
Terminated: False
Truncated: False


In [14]:
observation, info = env.reset(seed=SEED)
total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()                     # random action
    observation, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    steps += 1

    done = terminated or truncated
    if done:
        break

print(f"Episode finished after {steps} steps.")
print(f"Total reward: {total_reward}")
print(f"Succeeded (terminated early)? {terminated}")

Episode finished after 500 steps.
Total reward: -500.0
Succeeded (terminated early)? False


In [15]:
from IPython.display import HTML
from matplotlib import animation


def collect_random_episode_frames(seed=SEED, max_steps=500):
    """Run one episode with random actions and return the list of rendered frames."""
    render_env = gym.make("Acrobot-v1", render_mode="rgb_array")
    obs, info = render_env.reset(seed=seed)

    frames = [render_env.render()]
    for _ in range(max_steps):
        action = render_env.action_space.sample()
        _, _, terminated, truncated, _ = render_env.step(action)
        frames.append(render_env.render())
        if terminated or truncated:
            break

    render_env.close()
    return frames

def make_animation(frames, title=""):
    """Turn a list of image frames into an inline, playable animation."""
    fig, ax = plt.subplots()
    ax.axis("off")
    if title:
        ax.set_title(title)
    img = ax.imshow(frames[0])

    def update(i):
        img.set_data(frames[i])
        return [img]

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=40, blit=True
    )
    plt.close(fig)  # prevents a duplicate static image from also being displayed
    return anim

random_frames = collect_random_episode_frames()
print(f"Collected {len(random_frames)} frames from the random-action episode.")

random_anim = make_animation(random_frames, title="Random actions")
HTML(random_anim.to_jshtml())

Collected 501 frames from the random-action episode.
